# 1. Pipeline file check
Run all cells from top to bottom. This notebook checks source data, image coverage, preprocessing files, training outputs, and recommendation outputs. A failed later stage is normal if you have not run that stage yet.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config.yml').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)

In [ ]:
from IPython.display import display
from src.utils.pipeline_validation import validation_frame

report = validation_frame(PROJECT_ROOT / 'config.yml')
display(report.style.applymap(lambda value: 'color: #16803c; font-weight: bold' if value == 'PASS' else ('color: #c62828; font-weight: bold' if value == 'FAIL' else '')))
print(f"Passed: {(report.status == 'PASS').sum()} / {len(report)}")

The next cell gives useful dataset counts. The expected corrected selection is 3,164 outfits for the recovered MANGO CSVs.

In [ ]:
import pandas as pd
from src.utils.setup_utilities import load_config

config = load_config(PROJECT_ROOT / 'config.yml')
data = config['data']
products = pd.read_csv(data['products_path'], dtype={'cod_modelo_color': str})
outfits = pd.read_csv(data['outfits_path'], dtype={'cod_modelo_color': str})
selected = pd.read_csv(data['outfits_w_products_path'], dtype=str) if Path(data['outfits_w_products_path']).is_file() else pd.DataFrame()
summary = pd.Series({
    'catalog products': products.cod_modelo_color.nunique(),
    'source outfits': outfits.cod_outfit.nunique(),
    'selected outfits': len(selected),
    'image files': len(list((PROJECT_ROOT / 'data/images').glob('*.jpg'))),
})
display(summary.to_frame('count'))

Finally, inspect one generated training strip. Vertical lines mark the ten fixed product slots.

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from src.data_processing.image_processing import parse_tfrecord_fn
from src.data_processing.schema import PRODUCT_CLASSES

record_path = config['training']['train_dataset_path']
if Path(record_path).is_file():
    image = next(iter(tf.data.TFRecordDataset(record_path).map(parse_tfrecord_fn))).numpy()
    plt.figure(figsize=(20, 3))
    plt.imshow(image)
    for boundary in range(1, 10):
        plt.axvline(boundary * data['image_width'] - 0.5, color='red', linewidth=0.6)
    plt.xticks([(i + 0.5) * data['image_width'] for i in range(10)], PRODUCT_CLASSES, rotation=45, ha='right')
    plt.yticks([])
    plt.title('First training outfit strip')
    plt.show()
else:
    print('Training TFRecord is not available. Run the preprocessing pipeline first.')